# 57 — Adversarial evaluation dataset audit

**Purpose:** Quality control on the pair registry produced by **56**. No training; no final model comparison.

**Inputs:** `notebooks/results/adversarial_resume_pairs/`

**Outputs:**
- `notebooks/results/adversarial_eval_audit/`
- `figures/adversarial_eval_audit/`


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
import difflib

import numpy as np
import pandas as pd

_CWD = Path.cwd().resolve()
NOTEBOOK_DIR = _CWD if _CWD.name == "notebooks" else (_CWD / "notebooks")
REPO_ROOT = NOTEBOOK_DIR.parent
SRC = REPO_ROOT / "notebooks" / "results" / "adversarial_resume_pairs" / "adversarial_resume_pairs.csv"
RESULTS_DIR = REPO_ROOT / "notebooks" / "results" / "adversarial_eval_audit"
FIG_DIR = REPO_ROOT / "figures" / "adversarial_eval_audit"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not SRC.is_file():
    raise FileNotFoundError(f"Run 56 first: missing {SRC}")

df = pd.read_csv(SRC)
print("Rows:", len(df))

required = [
    "pair_id", "anchor_row_id", "supercategory", "attribute_edited",
    "anchor_resume_text", "counterfactual_resume_text",
]
for c in required:
    if c not in df.columns:
        raise KeyError(f"Missing column {c!r}")

df["anchor_resume_text"] = df["anchor_resume_text"].fillna("").astype(str)
df["counterfactual_resume_text"] = df["counterfactual_resume_text"].fillna("").astype(str)


def char_change_ratio(a: str, b: str) -> float:
    if not a:
        return 0.0
    return min(1.0, sum(1 for x, y in zip(a, b) if x != y) / max(len(a), 1))


def word_tokens(s: str) -> set[str]:
    return set(re.findall(r"[A-Za-z0-9]+", s.lower()))


df["len_anchor"] = df["anchor_resume_text"].str.len()
df["len_cf"] = df["counterfactual_resume_text"].str.len()
df["char_ratio_diff"] = (df["len_cf"] - df["len_anchor"]).abs() / df["len_anchor"].replace(0, np.nan)
df["approx_token_jaccard"] = df.apply(
    lambda r: len(word_tokens(r["anchor_resume_text"]) & word_tokens(r["counterfactual_resume_text"]))
    / max(1, len(word_tokens(r["anchor_resume_text"]))),
    axis=1,
)

# Broken generations: empty cf, identical to anchor, extreme shrink
issues = []
ident = (df["anchor_resume_text"] == df["counterfactual_resume_text"])
issues.extend([{"pair_id": r.pair_id, "issue": "identical_text"} for r in df.loc[ident].itertuples()])
empty_cf = df["counterfactual_resume_text"].str.len() == 0
issues.extend([{"pair_id": r.pair_id, "issue": "empty_counterfactual"} for r in df.loc[empty_cf].itertuples()])

audit = {
    "source_pairs_csv": str(SRC.relative_to(REPO_ROOT)),
    "n_pairs": int(len(df)),
    "n_unique_anchors": int(df["anchor_row_id"].nunique()),
    "attribute_counts": df["attribute_edited"].value_counts().to_dict(),
    "class_counts": df["supercategory"].value_counts().to_dict(),
    "validation_status_counts": df["validation_status"].value_counts().to_dict() if "validation_status" in df.columns else {},
    "median_char_len_anchor": float(df["len_anchor"].median()) if len(df) else 0.0,
    "median_abs_relative_len_change": float(df["char_ratio_diff"].median(skipna=True)) if len(df) else 0.0,
    "median_token_overlap_proxy": float(df["approx_token_jaccard"].median()) if len(df) else 0.0,
    "n_issue_identical": int(ident.sum()),
    "n_issue_empty_cf": int(empty_cf.sum()),
}

audit["clean_enough_for_model_eval"] = bool(
    len(df) > 0 and ident.sum() == 0 and empty_cf.sum() == 0
)

(RESULTS_DIR / "audit_summary.json").write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(issues).to_csv(RESULTS_DIR / "audit_issues.csv", index=False)
print(json.dumps(audit, indent=2))


Rows: 208
{
  "source_pairs_csv": "notebooks/results/adversarial_resume_pairs/adversarial_resume_pairs.csv",
  "n_pairs": 208,
  "n_unique_anchors": 208,
  "attribute_counts": {
    "city_location": 208
  },
  "class_counts": {
    "backend_general_dev": 67,
    "technical_specialized": 46,
    "web_frontend": 32,
    "sysadmin_devops_network": 25,
    "project_product": 17,
    "tech_support_helpdesk": 11,
    "generic_it_ops": 9,
    "it_governance_leadership": 1
  },
  "validation_status_counts": {
    "validated": 208
  },
  "median_char_len_anchor": 4308.0,
  "median_abs_relative_len_change": 0.0010482088540530547,
  "median_token_overlap_proxy": 0.994310088060088,
  "n_issue_identical": 0,
  "n_issue_empty_cf": 0,
  "clean_enough_for_model_eval": true
}


In [2]:
"""Example pairs (readable excerpts)."""
n_show = min(5, len(df))
examples = []
for i, r in df.head(n_show).iterrows():
    a = r["anchor_resume_text"][:900]
    c = r["counterfactual_resume_text"][:900]
    sm = difflib.unified_diff(
        a.splitlines(keepends=True), c.splitlines(keepends=True), lineterm="", n=2
    )
    diff_snip = "".join(list(sm)[:40])
    examples.append({
        "pair_id": r["pair_id"],
        "attribute_edited": r["attribute_edited"],
        "supercategory": r["supercategory"],
        "anchor_excerpt": a + ("…" if len(r["anchor_resume_text"]) > 900 else ""),
        "counterfactual_excerpt": c + ("…" if len(r["counterfactual_resume_text"]) > 900 else ""),
        "unified_diff_snippet": diff_snip,
    })

ex_df = pd.DataFrame(examples)
ex_path = RESULTS_DIR / "audit_example_pairs.csv"
ex_df.to_csv(ex_path, index=False)
print("Wrote", ex_path)
if len(ex_df):
    try:
        from IPython.display import display
        display(ex_df[["pair_id", "attribute_edited", "supercategory"]])
    except Exception:
        print(ex_df[["pair_id", "attribute_edited", "supercategory"]].to_string())


Wrote /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/adversarial_eval_audit/audit_example_pairs.csv


,pair_id,attribute_edited,supercategory
0,p_42d8450e0f083844,city_location,project_product
1,p_c66e2a0634631852,city_location,generic_it_ops
2,p_abbe650b6a8d66fa,city_location,web_frontend
3,p_b384550fb1f436ad,city_location,backend_general_dev
4,p_0a3a8ea71023fa1b,city_location,sysadmin_devops_network


In [3]:
"""Figures: attribute mix + overlap proxy."""
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 4))
    df["attribute_edited"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#54a24b")
    ax.set_title("57 — Pairs by attribute")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "audit_attribute_counts.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.hist(df["approx_token_jaccard"].clip(0, 1), bins=20, color="#b279a2")
    ax2.set_title("Token-overlap proxy (higher = more preserved)")
    ax2.set_xlabel("overlap proxy")
    fig2.tight_layout()
    fig2.savefig(FIG_DIR / "audit_token_overlap_hist.png", dpi=160, bbox_inches="tight")
    plt.close(fig2)
    print("Figures in", FIG_DIR)
except Exception as e:
    print("Figure skip:", e)


Figures in /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/adversarial_eval_audit


## Key takeaways

1. **Integrity:** The audit flags **identical anchor/counterfactual** rows and **empty** variants; high **token-overlap proxy** indicates non-sensitive text was largely preserved under the conservative rules.
2. **Class / attribute coverage:** Summary JSON reports **distribution** over `supercategory` and `attribute_edited` for scientific defensibility checks (imbalances are expected if some cues are rare).
3. **Fit for downstream:** If `clean_enough_for_model_eval` is true and spot checks look plausible, proceed to **58** for checkpoint evaluation. **Limitations:** Rule-based detectors **miss** many real-world sensitive cues; pronoun swaps can **misfire** on narrow contexts; header-name detection is **conservative** and will not cover all naming patterns.
